In [3]:
from ase.io import read, write
from ase import Atoms
import numpy as np
from sklearn.cluster import KMeans
import os

# =============================================
# 1. 读取 slab（Cartesian 坐标）
# =============================================
base_dir = "/home/ucaqzh0/thermol/100_co2_absorption"
slab_path = os.path.join(base_dir, "POSCAR")
slab = read(slab_path, format='vasp')
slab_positions = slab.get_positions()
z_positions = slab_positions[:, 2]

# ====================================================
# 2. 自动聚类 z 坐标，识别每一层
# ====================================================
n_atoms = len(slab)
max_layers = min(20, n_atoms)

best_k = None
best_inertia = 1e100

for k in range(2, max_layers):
    km = KMeans(n_clusters=k, n_init=10).fit(z_positions.reshape(-1,1))
    if km.inertia_ < best_inertia:
        best_inertia = km.inertia_
        best_k = k
        best_km = km

labels = best_km.labels_
centers = best_km.cluster_centers_.flatten()

# =============================================
# 3. 选择最高一层（表面）
# =============================================
sorted_layers = np.argsort(centers)
top_layer_id = sorted_layers[-1]           # 最顶部那一层
surface_z = centers[top_layer_id]

print(f"识别到 slab 共 {best_k} 层，最高一层 z = {surface_z:.3f} Å")

# =============================================
# 4. 构建 CO₂ 分子（在原点，对称结构）
# =============================================
d = 1.18   # C-O 键长
CO2 = Atoms(
    'OCO',
    positions=[
        (-d, 0, 0),
        (0,    0, 0),
        (d,  0, 0)
    ]
)

# =============================================
# 5. 将 CO₂ 放置到 slab 表面上方 ads_height（例如 3.0 Å）
# =============================================
ads_height = 3.5

xy_center = slab_positions[:, :2].mean(axis=0)

CO2.translate([
    xy_center[0],
    xy_center[1],
    surface_z + ads_height
])

# =============================================
# 6. 合并 slab 与 CO₂，并保留固定原子
# =============================================
constraints = slab.constraints
system = slab + CO2
if constraints:
    system.set_constraint(constraints)

# =============================================
# 7. 写出最终的 POSCAR（统一 Cartesian）
# =============================================
output_dir = os.path.join(base_dir, "physical_adsorption")
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, "POSCAR_CO2_ads.vasp")
write(output_path, system, format="vasp", vasp5=True)

print(f"✔ 已生成吸附模型：{output_path}")


识别到 slab 共 19 层，最高一层 z = 8.800 Å
✔ 已生成吸附模型：/home/ucaqzh0/thermol/100_co2_absorption/physical_adsorption/POSCAR_CO2_ads.vasp
